# Phase 2: Prompt Adaptation & LLM Pun Generation
**Project:** Spanglish, But Make It Punny (CMPT 419 / 983)  
**Authors:** Michaela Foo & Kevin Gonzalez  

### Notebook Objectives
1. Mount Google Drive and connect to the shared `CMPT419_983_Spanglish_Puns` project directory.
2. Inspect the cloned `Likhith-Asapu/Codemix-Pun-Generation` repository structure and load your finalized Phase 1 $(P_w, A_w, C_w)$ dataset (`spanglish_pun_pairs_final.csv` or `spanglish_pun_pairs_130_clean.csv`).
3. Define the **4 Spanglish few-shot prompting templates** adapted from Asapu et al. (2025):
   * **Method 1: Baseline** (Direct cross-lingual pun generation)
   * **Method 2: Contextually Aligned** (Anchoring the scenario around $C_w$ to activate $A_w$ before delivering $P_w$)
   * **Method 3: Question-Answer (QA)** (Setup question hinting at $A_w$/$C_w$ + punchline answer delivering $P_w$)
   * **Method 4: Subject-Masked** (Two-step mask-and-infill workflow replacing $A_w$ with $P_w$)
4. Run a **10-pair pilot test** to verify code-mixing naturalness before executing the full batch generation across all pairs with locked hyperparameters.
5. Export checkpointed CSVs for each method and compile the **100-item blinded evaluation sheet** (25 pairs $\times$ 4 methods) for Phase 3

## 1. Mount Google Drive & Initialize Project Paths
Connects to Google Drive, sets the working directory to `CMPT419_983_Spanglish_Puns`, and maps all subfolder paths (`03_processed_pairs/`, `04_repo_and_prompts/`, `05_generated_puns/`, `06_evaluation/`).

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CMPT419_983_Spanglish_Puns"

DIRS = {
    "raw": os.path.join(BASE_DIR, "01_raw_data"),
    "ipa": os.path.join(BASE_DIR, "02_ipa_dicts"),
    "processed": os.path.join(BASE_DIR, "03_processed_pairs"),
    "repo": os.path.join(BASE_DIR, "04_repo_and_prompts"),
    "generated": os.path.join(BASE_DIR, "05_generated_puns"),
    "eval": os.path.join(BASE_DIR, "06_evaluation"),
    "notebooks": os.path.join(BASE_DIR, "notebooks"),
}

for path in DIRS.values():
    os.makedirs(path, exist_ok=True)

os.chdir(BASE_DIR)
print("Working directory:", os.getcwd())
print("Files in 03_processed_pairs:", os.listdir(DIRS["processed"]))

Mounted at /content/drive
Working directory: /content/drive/MyDrive/CMPT419_983_Spanglish_Puns
Files in 03_processed_pairs: ['spanglish_pun_pairs_130_clean.csv']


## 2. Install SDK & Configure Gemini API Key
Installs the official `google-genai` Python SDK and loads your API key securely via Google Colab Secrets (`userdata.get('GEMINI_API_KEY')`).  
*Tip: In Colab, click the **Key icon (Secrets)** in the left sidebar, add a secret named `GEMINI_API_KEY`, and toggle "Notebook access" ON.*

In [ ]:
!pip install -q -U google-genai pandas tqdm

import json
import time
import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types
from google.colab import userdata

# Load API key from Colab Secrets (fallback to environment variable if needed)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    raise ValueError("Please add GEMINI_API_KEY in Colab Secrets (left sidebar key icon) before continuing.")

client = genai.Client(api_key=GEMINI_API_KEY)

# Lock model and generation settings across all 4 prompting methods for a fair comparison
MODEL_ID = "gemini-2.5-flash"
GEN_CONFIG = types.GenerateContentConfig(
    temperature=0.7,
    top_p=0.95,
    max_output_tokens=200,
)

print(f"Configured client with model: {MODEL_ID} (temperature={GEN_CONFIG.temperature})")

Configured client with model: gemini-2.5-flash (temperature=0.7)


## 3. Inspect Reference Repo & Load Finalized Phase 1 Pun Pairs
Checks the cloned `Codemix-Pun-Generation` directory and loads your finalized $(P_w, A_w, C_w)$ dataset (automatically preferring `spanglish_pun_pairs_final.csv` looking for manually verified file, otherwise falling back to `spanglish_pun_pairs_130_clean.csv`).

In [ ]:
repo_dir = os.path.join(DIRS["repo"], "Codemix-Pun-Generation")
if os.path.exists(repo_dir):
    print("Contents of Codemix-Pun-Generation repo:")
    for root, dirs, files in os.walk(repo_dir):
        level = root.replace(repo_dir, "").count(os.sep)
        if level <= 2 and ".git" not in root:
            indent = "  " * level
            print(f"{indent}{os.path.basename(root)}/")
            for f in files:
                print(f"{indent}  - {f}")

# Load finalized Phase 1 dataset
final_path = os.path.join(DIRS["processed"], "spanglish_pun_pairs_final.csv")
clean_path = os.path.join(DIRS["processed"], "spanglish_pun_pairs_130_clean.csv")

dataset_path = final_path if os.path.exists(final_path) else clean_path
pairs_df = pd.read_csv(dataset_path)

print(f"\nLoaded {len(pairs_df)} pun pairs from: {os.path.basename(dataset_path)}")
display(pairs_df[["pair_id", "spanish_word", "english_word", "context_word", "phonetic_similarity"]].head(10))

Contents of Codemix-Pun-Generation repo:
Codemix-Pun-Generation/
  - Codemix_Pun_Generation.pdf
  - README.md
  - requirements.txt
  - samples.json
  automated_pipeline/
    - add_meaning.py
    - annotated_data.json
    - annotated_pipeline_data.json
    - annotation_refactor.py
    - bert_embedding.py
    - calculate_scores.py
    - combine_files.py
    - compatibility_model_2.pth
    - compatibility_scores.json
    - compatibility_scores.png
    - english_words.csv
    - fast_siamese.py
    - fastext.py
    - fasttext_compatibility_model.pth
    - generate_compatibility_score.py
    - generate_ipa.py
    - generate_pairs.py
    - generate_words.py
    - hindi_words.csv
    - pipeline_puns.json
    - pun_alternate_word.json
    - pun_success.py
    - siamese.py
    - siamese_compatibility_model.pth
    - train_mbert.py
    - train_nn.py
  baseline/
    - Chatgpt_pun_gpt.json
    - generate_pun_chatgpt.py
  classification/
    - .DS_Store
    - attention_weights_1.png
    - batch_scri

,pair_id,spanish_word,english_word,context_word,phonetic_similarity
0,1,sopa,soap,"cleansing agent, cleanser, cleaner",0.83
1,2,flor,floor,"horizontal surface, level, inside",0.80
2,3,burro,burrow,"hole, hollow, animal",0.75
3,4,gato,ghetto,"quarter, formerly, restricted",0.75
4,5,sol,soul,"spirit, immaterial, part",0.80
5,6,pez,pets,"animal, animate being, beast",0.80
6,7,pan,pun,"fun, play, sport",0.75
7,8,vino,beano,"board game, game, numbered",0.75
8,9,pelo,pillow,"cushion, support, head",0.70
9,10,velo,willow,"tree, numerous, deciduous",0.65


## 4. Define & Save the 4 Adapted Spanglish Few-Shot Prompt Templates
Implements the 4 prompting strategies from **Asapu et al. (2025)** adapted to Spanish-English code-mixing:
1. **Baseline:** Direct instruction with 4 Spanglish few-shot examples using $(P_w, A_w)$.
2. **Contextually Aligned:** Uses $(P_w, A_w, C_w)$ to build a setup strongly tied to the English semantic context ($C_w$) so the reader expects $A_w$ before encountering the Spanish homophone $P_w$.
3. **Question-Answer (QA):** Formats the pun as a setup question (`Q:`) hinting at $A_w$/$C_w$ and a punchline (`A:`) delivering $P_w$.
4. **Subject-Masked:** Two-stage generation—Stage A generates a natural English sentence using $A_w$ and $C_w$ with $A_w$ masked as `[MASK]`; Stage B infills `[MASK]` with the Spanish word $P_w$ and blends natural Spanglish framing around it.

*(Note: All few-shot examples below use pairs excluded from your test CSV so there is zero data leakage.)*

In [ ]:
PROMPT_TEMPLATES = {
    "baseline": {
        "system": (
            "You are a witty bilingual comedian fluent in Spanglish (Spanish-English code-mixing). "
            "Your task is to write ONE short, funny, natural-sounding Spanglish sentence that creates a "
            "phonetic pun by using a Spanish word (Pw) in place of a similar-sounding English word (Aw). "
            "Output ONLY the final Spanglish pun sentence—no explanations or translations."
        ),
        "few_shot": [
            {
                "input": "Spanish Pun Word (Pw): crema ('cream') | English Alternate Word (Aw): karma ('destiny/fate')",
                "output": "Don't worry about him cheating on the test—what goes around comes around, es puro crema."
            },
            {
                "input": "Spanish Pun Word (Pw): beso ('kiss') | English Alternate Word (Aw): peso ('currency/weight')",
                "output": "I tried paying for my tacos with affection, pero the cashier said she couldn't accept a single beso."
            },
            {
                "input": "Spanish Pun Word (Pw): burro ('donkey') | English Alternate Word (Aw): burrow ('animal tunnel')",
                "output": "The rabbit dug all night in my backyard until he finally found a cozy little burro."
            },
            {
                "input": "Spanish Pun Word (Pw): sopa ('soup') | English Alternate Word (Aw): soap ('cleaning bar')",
                "output": "After dropping my spoon in the broth, I had to wash my hands with a bar of sopa."
            }
        ]
    },

    "contextually_aligned": {
        "system": (
            "You are a witty bilingual comedian fluent in Spanglish (Spanish-English code-mixing). "
            "You are given a Spanish Pun Word (Pw), a similar-sounding English Alternate Word (Aw), and "
            "English Context Keywords (Cw) related to Aw. Build a natural scenario grounded in the context Cw "
            "so the listener expects the English word Aw, then pivot into Spanish and deliver Pw as the punchline. "
            "Output ONLY the final Spanglish pun sentence."
        ),
        "few_shot": [
            {
                "input": "Pw: crema ('cream') | Aw: karma | Cw: justice, fate, payback",
                "output": "When the bully slipped on his own spilled latte, abuela smiled and said, 'Eso es instant crema.'"
            },
            {
                "input": "Pw: beso ('kiss') | Aw: peso | Cw: money, currency, bank",
                "output": "Inflation in Mexico is getting wild—ahora you can't even buy chicle without spending a whole beso."
            },
            {
                "input": "Pw: burro ('donkey') | Aw: burrow | Cw: rabbit, underground, tunnel",
                "output": "The prairie dog spent all morning digging underground just to build a stubborn little burro."
            },
            {
                "input": "Pw: sopa ('soup') | Aw: soap | Cw: shower, bubbles, clean",
                "output": "I jumped in the shower after cooking caldo and accidentally scrubbed my face with a bar of sopa."
            }
        ]
    },

    "question_answer": {
        "system": (
            "You are a witty bilingual comedian fluent in Spanglish (Spanish-English code-mixing). "
            "Write a short, funny Question-and-Answer joke in Spanglish. The Question (Q:) must set up a scenario "
            "based on the English Context Keywords (Cw) and Alternate Word (Aw), and the Answer (A:) must deliver "
            "the punchline using the similar-sounding Spanish Pun Word (Pw). Output ONLY 'Q: ... A: ...'."
        ),
        "few_shot": [
            {
                "input": "Pw: crema ('cream') | Aw: karma | Cw: justice, fate, payback",
                "output": "Q: What do you call cosmic payback at a Mexican coffee shop? A: Instant crema."
            },
            {
                "input": "Pw: beso ('kiss') | Aw: peso | Cw: money, currency, bank",
                "output": "Q: Why did the romantic bring his girlfriend to the currency exchange? A: Because he wanted to give her a Mexican beso."
            },
            {
                "input": "Pw: burro ('donkey') | Aw: burrow | Cw: rabbit, underground, tunnel",
                "output": "Q: Where does a stubborn rabbit sleep underground? A: Inside his own little burro."
            },
            {
                "input": "Pw: sopa ('soup') | Aw: soap | Cw: shower, bubbles, clean",
                "output": "Q: What do you use to wash your hands in an abuela's kitchen? A: A fresh bar of sopa."
            }
        ]
    },

    "subject_masked_stage1": {
        "system": (
            "Write ONE natural, everyday English sentence (10-16 words) that uses the given English target word (Aw) "
            "together with its context keywords (Cw). Replace ONLY the target word Aw with '[MASK]'. "
            "Output ONLY the masked English sentence."
        ),
        "few_shot": [
            {
                "input": "Aw: karma | Cw: justice, fate, payback",
                "output": "After he stole my parking spot and got a ticket, I knew [MASK] was real."
            },
            {
                "input": "Aw: burrow | Cw: rabbit, underground, tunnel",
                "output": "The tired rabbit crawled deep underground into its warm [MASK] for the night."
            },
            {
                "input": "Aw: soap | Cw: shower, bubbles, clean",
                "output": "I reached into the shower caddy and grabbed a fresh bar of [MASK]."
            }
        ]
    },

    "subject_masked_stage2": {
        "system": (
            "You are a witty bilingual comedian fluent in Spanglish (Spanish-English code-mixing). "
            "You are given an English sentence containing '[MASK]' (where the English word Aw originally belonged) "
            "and a similar-sounding Spanish Pun Word (Pw). Replace '[MASK]' with Pw and naturally blend a few "
            "Spanish words or particles into the sentence so it reads like authentic Spanglish humor. "
            "Output ONLY the final Spanglish sentence."
        ),
        "few_shot": [
            {
                "input": "Masked Sentence: After he stole my parking spot and got a ticket, I knew [MASK] was real. | Pw: crema ('cream') | Aw: karma",
                "output": "Después de robarme el parking spot y recibir un ticket, I knew crema was real."
            },
            {
                "input": "Masked Sentence: The tired rabbit crawled deep underground into its warm [MASK] for the night. | Pw: burro ('donkey') | Aw: burrow",
                "output": "El conejo cansado crawled deep underground into su warm burro for the night."
            },
            {
                "input": "Masked Sentence: I reached into the shower caddy and grabbed a fresh bar of [MASK]. | Pw: sopa ('soup') | Aw: soap",
                "output": "Me metí al shower y agarré a fresh bar of sopa."
            }
        ]
    }
}

# Save templates to 04_repo_and_prompts/ for reproducibility
templates_path = os.path.join(DIRS["repo"], "spanglish_prompt_templates.json")
with open(templates_path, "w", encoding="utf-8") as f:
    json.dump(PROMPT_TEMPLATES, f, ensure_ascii=False, indent=2)

print("Saved Spanglish prompt templates to:", templates_path)

Saved Spanglish prompt templates to: /content/drive/MyDrive/CMPT419_983_Spanglish_Puns/04_repo_and_prompts/spanglish_prompt_templates.json


## 5. Define Generation Functions for All 4 Prompting Methods
Builds clean helper functions that format few-shot prompts, call the Gemini API with exponential backoff retry logic, and handle the two-stage **Subject-Masked** infill workflow.

In [ ]:
def format_few_shot_prompt(template_key, user_query):
    tpl = PROMPT_TEMPLATES[template_key]
    examples_str = "\n\n".join(
        [f"Input: {ex['input']}\nOutput: {ex['output']}" for ex in tpl["few_shot"]]
    )
    return f"{tpl['system']}\n\n### Examples\n{examples_str}\n\n### Task\nInput: {user_query}\nOutput:"

def call_llm(prompt_text, max_retries=4, base_delay=2.0):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt_text,
                config=GEN_CONFIG
            )
            return response.text.strip().replace("\n", " ")
        except Exception as e:
            if attempt == max_retries - 1:
                return f"[ERROR: {str(e)}]"
            time.sleep(base_delay * (2 ** attempt))

def generate_for_row(row, method):
    pw = row["spanish_word"]
    aw = row["english_word"]
    cw = row["context_word"]
    es_mean = str(row["spanish_meaning"]).split(";")[0][:40]
    en_mean = str(row["english_meaning"]).split(";")[0][:40]

    if method == "baseline":
        query = f"Spanish Pun Word (Pw): {pw} ('{es_mean}') | English Alternate Word (Aw): {aw} ('{en_mean}')"
        return call_llm(format_few_shot_prompt("baseline", query)), ""

    elif method == "contextually_aligned":
        query = f"Pw: {pw} ('{es_mean}') | Aw: {aw} ('{en_mean}') | Cw: {cw}"
        return call_llm(format_few_shot_prompt("contextually_aligned", query)), ""

    elif method == "question_answer":
        query = f"Pw: {pw} ('{es_mean}') | Aw: {aw} ('{en_mean}') | Cw: {cw}"
        return call_llm(format_few_shot_prompt("question_answer", query)), ""

    elif method == "subject_masked":
        # Stage 1: Generate masked English context sentence
        q1 = f"Aw: {aw} ('{en_mean}') | Cw: {cw}"
        masked_sent = call_llm(format_few_shot_prompt("subject_masked_stage1", q1))
        if "[MASK]" not in masked_sent:
            masked_sent = re.sub(rf"\b{re.escape(aw)}\b", "[MASK]", masked_sent, flags=re.IGNORECASE)

        # Stage 2: Infill [MASK] with Spanish Pw and adapt to Spanglish
        q2 = f"Masked Sentence: {masked_sent} | Pw: {pw} ('{es_mean}') | Aw: {aw}"
        final_sent = call_llm(format_few_shot_prompt("subject_masked_stage2", q2))
        return final_sent, masked_sent

    else:
        raise ValueError(f"Unknown method: {method}")

## 6. Run 10-Pair Pilot Test (Sanity Check Before Full Batch)
Generates outputs across all 4 prompting methods on the first **10 pun pairs** and saves them to `05_generated_puns/pilot_10_pairs.csv`. Inspect the outputs below to confirm that:
1. Every generated sentence actually includes the target Spanish word ($P_w$).
2. The sentences naturally code-switch between Spanish and English rather than defaulting to monolingual English.

In [ ]:
import re

pilot_df = pairs_df.head(10).copy()
methods = ["baseline", "contextually_aligned", "question_answer", "subject_masked"]

pilot_records = []
for _, row in tqdm(pilot_df.iterrows(), total=len(pilot_df), desc="Running 10-Pair Pilot"):
    rec = {
        "pair_id": row["pair_id"],
        "spanish_word": row["spanish_word"],
        "english_word": row["english_word"],
        "context_word": row["context_word"],
        "phonetic_similarity": row["phonetic_similarity"],
    }
    for m in methods:
        out_text, masked_intermediate = generate_for_row(row, m)
        rec[m] = out_text
        if m == "subject_masked":
            rec["subject_masked_template"] = masked_intermediate
        time.sleep(0.3)
    pilot_records.append(rec)

pilot_out_df = pd.DataFrame(pilot_records)
pilot_path = os.path.join(DIRS["generated"], "pilot_10_pairs.csv")
pilot_out_df.to_csv(pilot_path, index=False)

print(f"Pilot results saved to: {pilot_path}")
display(pilot_out_df[["spanish_word", "english_word", "baseline", "contextually_aligned", "question_answer", "subject_masked"]])

## 7. Run Full Batch Generation Across All Pairs (Checkpointed by Method)
Runs batch LLM generation across all finalized pun pairs (`pairs_df`) for each of the 4 prompting methods. Each method is checkpointed immediately to its own CSV in `05_generated_puns/` (`batch_baseline.csv`, `batch_contextually_aligned.csv`, `batch_question_answer.csv`, `batch_subject_masked.csv`) so progress is preserved if a Colab session disconnects.

In [ ]:
methods = ["baseline", "contextually_aligned", "question_answer", "subject_masked"]
all_method_dfs = {}

for method in methods:
    out_file = os.path.join(DIRS["generated"], f"batch_{method}.csv")

    # Resume from checkpoint if file already exists
    if os.path.exists(out_file):
        existing_df = pd.read_csv(out_file)
        done_ids = set(existing_df["pair_id"])
        records = existing_df.to_dict("records")
        print(f"[{method}] Resuming from checkpoint: {len(done_ids)}/{len(pairs_df)} already generated.")
    else:
        done_ids = set()
        records = []

    remaining_df = pairs_df[~pairs_df["pair_id"].isin(done_ids)]

    for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc=f"Generating: {method}"):
        generated_pun, masked_step = generate_for_row(row, method)

        # Automatic check: verify Pw appears in the generated sentence
        contains_pw = bool(re.search(rf"\b{re.escape(str(row['spanish_word']))}\b", generated_pun, flags=re.IGNORECASE))

        rec = {
            "pair_id": row["pair_id"],
            "method": method,
            "spanish_word": row["spanish_word"],
            "english_word": row["english_word"],
            "context_word": row["context_word"],
            "phonetic_similarity": row["phonetic_similarity"],
            "generated_pun": generated_pun,
            "contains_pw": contains_pw,
            "masked_template": masked_step
        }
        records.append(rec)

        # Save incremental checkpoint every 10 rows
        if len(records) % 10 == 0:
            pd.DataFrame(records).to_csv(out_file, index=False)
        time.sleep(0.35)

    method_df = pd.DataFrame(records)
    method_df.to_csv(out_file, index=False)
    all_method_dfs[method] = method_df
    pw_rate = method_df["contains_pw"].mean() * 100
    print(f"[{method}] Complete! Saved {len(method_df)} rows | Pw inclusion rate: {pw_rate:.1f}%\n")

# Combine all 4 methods into one master file
master_df = pd.concat(all_method_dfs.values(), ignore_index=True)
master_path = os.path.join(DIRS["generated"], "all_generated_puns_master.csv")
master_df.to_csv(master_path, index=False)
print("Saved combined master file:", master_path)

## 8. Sample & Export 100 Blinded Sentences for Phase 3 Human Evaluation
Follows your proposal's Phase 3 sampling plan:
* Randomly samples **25 pun pairs** (preferring pairs where all 4 methods successfully included $P_w$) $\times$ **4 methods** = **100 sentences**.
* Shuffles the 100 sentences with a fixed random seed (`random_state=42`) and assigns an anonymous `sentence_id` (`S001`–`S100`).
* Exports two files into `06_evaluation/`:
  1. `eval_100_blinded_for_raters.csv` — Contains **no method labels**, ready to paste into Google Forms/Sheets with columns for **Pun Success (0/1)**, **Funniness (1–5)**, and **Acceptability (1–5)**.
  2. `eval_100_answer_key.csv` — Private mapping linking each `sentence_id` back to its generating method (`baseline`, `contextually_aligned`, `question_answer`, `subject_masked`) for Phase 3 analysis.

In [ ]:
# Identify pairs where all 4 methods successfully included the Spanish pun word Pw
valid_counts = master_df.groupby("pair_id")["contains_pw"].sum()
complete_pair_ids = valid_counts[valid_counts == 4].index.tolist()

print(f"Pairs where all 4 methods included Pw: {len(complete_pair_ids)} / {len(pairs_df)}")

# Sample 25 pairs (25 pairs x 4 methods = 100 evaluation sentences)
sample_pool = complete_pair_ids if len(complete_pair_ids) >= 25 else pairs_df["pair_id"].tolist()
sampled_ids = pd.Series(sample_pool).sample(n=min(25, len(sample_pool)), random_state=42).tolist()

eval_subset = master_df[master_df["pair_id"].isin(sampled_ids)].copy()

# Shuffle rows so raters cannot guess the prompting method by ordering
eval_shuffled = eval_subset.sample(frac=1.0, random_state=42).reset_index(drop=True)
eval_shuffled.insert(0, "sentence_id", [f"S{i:03d}" for i in range(1, len(eval_shuffled) + 1)])

# 1. Save full answer key (kept private for Phase 3 scoring)
key_path = os.path.join(DIRS["eval"], "eval_100_answer_key.csv")
eval_shuffled.to_csv(key_path, index=False)

# 2. Save blinded rating sheet for human annotators
blinded_df = eval_shuffled[[
    "sentence_id",
    "spanish_word",
    "english_word",
    "generated_pun"
]].copy()
blinded_df["pun_success_0_1"] = ""
blinded_df["funniness_1_5"] = ""
blinded_df["acceptability_1_5"] = ""
blinded_df["error_category_optional"] = ""

blinded_path = os.path.join(DIRS["eval"], "eval_100_blinded_for_raters.csv")
blinded_df.to_csv(blinded_path, index=False)

print(f"Exported {len(blinded_df)} blinded sentences to:\n  - {blinded_path}\n  - {key_path}")
display(blinded_df.head(10))